In [0]:
-- ============================================================================
-- Gold Layer Tables for Sales Analytics
-- ============================================================================

-- Create daily revenue summary table
CREATE OR REPLACE TABLE sales.gold.revenue_by_day AS
SELECT 
  transaction_date AS `date`,
  SUM(total_spent) AS `total`,
  COUNT(*) AS `amount_of_transactions`
FROM sales.gold.sales_fact
GROUP BY transaction_date;

SELECT * FROM sales.gold.revenue_by_day;


-- Create best seller table showing the top-selling item each day
CREATE OR REPLACE TABLE sales.gold.best_seller_of_each_day AS
WITH ranked AS (
  SELECT
    transaction_date AS `date`,
    item_id AS `item`,
    quantity,
    RANK() OVER (PARTITION BY transaction_date ORDER BY quantity DESC) AS pos
  FROM sales.gold.sales_fact
)

SELECT date, item, quantity
FROM ranked
WHERE pos = 1;

SELECT * FROM sales.gold.best_seller_of_each_day;


-- This table identifies the top ten active customers in each city based on their total spending
CREATE OR REPLACE TABLE sales.gold.top_ten_active_customers_from_each_city AS
WITH ranked_total_spent_by_active_user AS (
  SELECT
    c.email,
    c.city,
    SUM(f.total_spent) AS total,
    RANK() OVER (PARTITION BY c.city ORDER BY SUM(f.total_spent) DESC) AS pos
  FROM sales.gold.sales_fact f
  JOIN sales.gold.customers_dim c ON f.customer_id = c.customer_id
  WHERE c.is_active = true
  GROUP BY c.customer_id, c.email, c.city
)

SELECT email, city, total
FROM ranked_total_spent_by_active_user
WHERE pos <= 10;

SELECT * FROM sales.gold.top_ten_active_customers_from_each_city
ORDER BY city, total DESC;
